In [1]:
%run functions_dbs.py
from scipy import stats

fs = 12
%matplotlib

Using matplotlib backend: MacOSX


Load data

In [2]:
file = 'dataPrep/Burggraben 2021 Oktober O2 and pH.xls'
SoftwareFile = input('is it a measurement file directly from the software? ')

is it a measurement file directly from the software? True


In [3]:
if SoftwareFile == 'True':
    # raw measurement file pre-processed and saved per default as rawData file
    dsheets = _loadFile4GUI(file=file)
else:
    # old version with pre-processed files:
    dsheets = pd.read_excel(file, sheet_name=None)

print('available sheet names', list(dsheets.keys()))

available sheet names ['O2_all', 'pH_all']


In [4]:
sheet_select = input('what is the sheet name? --> ')

what is the sheet name? --> O2_all


Define parameter for analysis

In [5]:
do2_res = dict()
dO2_core, dic_dcore, dhiden_object, dpen_all, dfig_pen = dict(), dict(), dict(), dict(), dict()

steps = 0.05                     # 
gmod = Model(_gompertz_curve)    # define model for curve fit
plot_inter = True               # shall we plot the interim steps as well? (for control)
type_fig = ['png', 'jpg']

baseline shift

In [6]:
if SoftwareFile == 'True':
    ddata = dsheets[sheet_select]
else:
    ddata = dsheets[sheet_select].set_index('Nr')
    
# pre-set of parameters
gmod = Model(_gompertz_curve)

# ----------------------------------------------------------------------------------
# list all available cores for O2 sheet
ls_core = list(dict.fromkeys(ddata['Core'].to_numpy()))

# import all measurements for given parameter
dic_dcore, ls_nr, ls_colname = load_measurements(dsheets=ddata, ls_core=ls_core, para=sheet_select)

# curve fit and baseline finder
dfit, dic_deriv = fit_baseline(ls_core=ls_core, ls_nr=ls_nr, dic_dcore=dic_dcore, steps=steps, gmod=gmod)

# baseline shift
data_shift = baseline_shift(dic_dcore=dic_dcore, dic_deriv=dic_deriv)

# ------------------------------------------------------------------------------------
# plotting interim results
dfig = plot_interimsteps(dfit=dfit, dic_dcore=dic_dcore, dic_deriv=dic_deriv, 
                         ls_core=ls_core, data_shift=data_shift, plot_interim=plot_inter)

/Users/au652733/Python/Project_julia/functions_dbs.py:54: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all axes decorations. 
  plt.tight_layout()
/Users/au652733/Python/Project_julia/functions_dbs.py:59: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig, ax = plt.subplots(figsize=figsize)


additional options for data manipulation

In [8]:
%run functions_dbs.py

In [10]:
# import all measurements for given parameter
dic_dcore, ls_nr, ls_colname = load_measurements(dsheets=ddata, ls_core=ls_core, para=sheet_select)

# curve fit and baseline finder
dfit, dic_deriv = fit_baseline(ls_core=ls_core, ls_nr=ls_nr, dic_dcore=dic_dcore, steps=steps, gmod=gmod)

In [11]:
# train code based on an example - plot_interimsteps
core = ls_core[0]
ls_cropx, ls_out = list(), list()
# baseline shift
nr = 1

print('red chi2:', dfit[core][nr][0].redchi)

# -----------------------------------------------------------
fig, ax = plot_surfacefinder(dic_dcore=dic_dcore, dfit=dfit, core=core, nr=nr, dic_deriv=dic_deriv)
def onclick(event):
    if event.key == 'enter':
        print('outlier detection')
        ls_out.append(event.xdata)
    else:
        print('crop data')
        ls_cropx.append(event.xdata)
cid = fig.canvas.mpl_connect('button_press_event', onclick)
plt.show()

red chi2: 21.785140334033137


In [12]:
def plot_surfacefinder_DF(dic_dcore, dfit, dic_deriv, figsize=(5, 3)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.canvas.set_window_title('Fit profile data - core {} #{}'.format(core, nr))
    ax1 = ax.twinx()
    ax.set_xlabel('Depth [µm]'), ax.set_ylabel('O2 [mV]'), ax1.set_ylabel('1st derivative', color='#0077b6')

    ax.plot(dic_dcore.index, dic_dcore['O2_mV'], lw=0, marker='o', ms=4, color='k')
    ax.plot(dfit[dfit.columns[0]], lw=0.75, ls=':', color='k')

    ax1.plot(dic_deriv, lw=1., color='#0077b6')
    ax1.axvline(dic_deriv.idxmin().values[0], ls='-.', color='darkorange', lw=1.5)
    text = 'surface level \nat {:.1f}µm'
    ax.text(dic_deriv.idxmin().values[0]*2, dic_dcore['O2_mV'].max()*1.15,
            text.format(dic_deriv.idxmin().values[0]), ha="left", va="center", color='darkorange', size=9.5)

    sns.despine()
    ax.spines['right'].set_visible(True)
    plt.tight_layout()
    return fig, ax


In [13]:
# data manipulation 
print('dataframe to crop to depth:', ls_cropx)

# 1) zoom to area (min, max x-values)
if ls_cropx:
    dcore_crop = dic_dcore[core][nr].loc[min(ls_cropx): max(ls_cropx)]
else:
    dcore_crop = dic_dcore[core][nr]
if ls_out:
    ls_pop = [min(dcore_crop.index.to_numpy(), key=lambda x:abs(x-ls_out[p])) for p in range(len(ls_out))]
    # drop in case value is still there
    for p in ls_pop:
        if p in dcore_crop.index:
            dcore_crop.drop(p, inplace=True)

# re-do fitting - curve fit and baseline finder
res, df_fit_crop, df_fitder = baseline_finder_DF(dic_dcore=dcore_crop, steps=steps, model=gmod)

# re-draw plot
fig, ax = plot_surfacefinder_DF(dic_dcore=dcore_crop, dfit=df_fit_crop, dic_deriv=df_fitder)
plt.show()

print('red chi2:', res.redchi)

dataframe to crop to depth: []
red chi2: 21.785140334033137


----------------------------------------------------------------

In [13]:
file = 'Complete_T2.xlsx'
file_calib = 'O2_depthProfile/gastables.pdf'

# when executing this function - there is no possibility (yet) to select the curves in the pre-plot.
dresults = O2_depthProfile(file, file_calib=file_calib, temp_degC=13, salinity=0, O2_pen=5, unit='µmol/l', lim=150,
                           steps=0.5, plot_inter=False)

KeyError: 'Core'

maximal dissolved O2

In [14]:
T = 13.2 # degC
salinity = 0 # o/oo

In [16]:
pdO2 = dict({'A1': -173.4292, 'A2': 249.6339, 'A3': 143.3483, 'A4': -21.8492, 
             'B1': -0.033096, 'B2': 0.014259, 'B3': -0.0017000, 
             'STP': 22.4139, 'R': 0.0821})

tempK = T + 273.15
convF = 100/tempK
# taylor series expansion (Taylor) with specific constants for O2 
taylor = np.exp(pdO2['A1'] + pdO2['A2']*convF + pdO2['A3']*np.log(1/convF) + 
             pdO2['A4']*(1/convF) +
             salinity*(pdO2['B1']+pdO2['B2']*(1/convF)+pdO2['B3']*(1/convF)**2))

dO2_max = taylor/(pdO2['R']*273.15)*1000 # µM
dO2_max

326.971804527664